# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [2]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

SEED=1

In [3]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [4]:
fraud.shape

(1000000, 8)

In [5]:
fraud.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   distance_from_home              1000000 non-null  float64
 1   distance_from_last_transaction  1000000 non-null  float64
 2   ratio_to_median_purchase_price  1000000 non-null  float64
 3   repeat_retailer                 1000000 non-null  float64
 4   used_chip                       1000000 non-null  float64
 5   used_pin_number                 1000000 non-null  float64
 6   online_order                    1000000 non-null  float64
 7   fraud                           1000000 non-null  float64
dtypes: float64(8)
memory usage: 61.0 MB


In [6]:
fraud.describe()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000
mean,26.628792,5.036519,1.824182,0.881536,0.350399,0.100608,0.650552,0.087403
std,65.390784,25.843093,2.799589,0.323157,0.477095,0.300809,0.476796,0.282425
min,0.004874,0.000118,0.004399,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.878008,0.296671,0.475673,1.000000,0.000000,0.000000,0.000000,0.000000
50%,9.967760,0.998650,0.997717,1.000000,0.000000,0.000000,1.000000,0.000000
75%,25.743985,3.355748,2.096370,1.000000,1.000000,0.000000,1.000000,0.000000
max,10632.723672,11851.104565,267.802942,1.000000,1.000000,1.000000,1.000000,1.000000


In [7]:
fraud["fraud"].value_counts(normalize=True)

fraud
0.0    0.912597
1.0    0.087403
Name: proportion, dtype: float64

In [8]:
# Finding 1 - The dataset is heavily imbalanced.
# 91.3% of transactions are legitimate (0) and only 8.7% are fraudulent (1).
# This means the model could achieve 91% accuracy by simply predicting "not fraud" every time,
# This would be useless in practice. We need to handle this imbalance.

In [9]:
y = fraud["fraud"]
X = fraud.drop("fraud", axis=1)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=SEED)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

X_train shape: (800000, 7)
X_test shape:  (200000, 7)


In [11]:
scaler = StandardScaler()
scaler.set_output(transform="pandas")

# Fit ONLY on X_train --> avoids leaking any information from the test set into the scaling parameters
X_train_scaled_df = scaler.fit_transform(X_train)

# Then apply that same transformation to X_test
X_test_scaled_df = scaler.transform(X_test)

# BEFORE: original scale
print('BEFORE scaling (X_train):')
print(X_train.describe().loc[['mean', 'std']].round(4))
print()

# AFTER: standardized scale
print('AFTER scaling (X_train):')
print(X_train_scaled_df.describe().loc[['mean', 'std']].round(4))

BEFORE scaling (X_train):
      distance_from_home  distance_from_last_transaction  \
mean             26.6840                          5.0569   
std              66.2604                         26.9935   

      ratio_to_median_purchase_price  repeat_retailer  used_chip  \
mean                          1.8230           0.8814     0.3509   
std                           2.7831           0.3233     0.4773   

      used_pin_number  online_order  
mean           0.1007        0.6502  
std            0.3009        0.4769  

AFTER scaling (X_train):
      distance_from_home  distance_from_last_transaction  \
mean                 0.0                             0.0   
std                  1.0                             1.0   

      ratio_to_median_purchase_price  repeat_retailer  used_chip  \
mean                             0.0              0.0        0.0   
std                              1.0              1.0        1.0   

      used_pin_number  online_order  
mean              0.0   

In [12]:
log = LogisticRegression(max_iter=1000)
model = log.fit(X_train_scaled_df, y_train)

print("Model trained!")

Model trained!


In [13]:
# Evaluate the model on both Train and Test, to check for overfitting
y_pred_train = model.predict(X_train_scaled_df)     # train
y_pred_test = model.predict(X_test_scaled_df)       # test

accuracy_train = accuracy_score(y_train, y_pred_train)  # train, just to compare with the report below
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()

# Classification Report (Test)
# ----------------------------
print(classification_report(y_test, y_pred_test, target_names=['No Fraud', 'Fraud'], zero_division=0))

Accuracy (Train): 95.88%

              precision    recall  f1-score   support

    No Fraud       0.96      0.99      0.98    182383
       Fraud       0.90      0.60      0.72     17617

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.95    200000



In [14]:
# Analysis 1
# Model 1 - Logistic Regression on Imbalanced Data
# Accuracy: 95.88% - looks high but is misleading due to class imbalance. 91% of the data was legit already. 
# No Fraud (class 0): Recall = 0.99 - model correctly identifies 99% of legit transactions
# Fraud (class 1): Recall = 0.60 - model misses 40% of actual fraud cases
# Precision for Fraud = 0.90 - when it does flag fraud, it is right 90% of the time
# Key issue: Low recall for fraud means 4 out of 10 real fraud cases go undetected.
# Root cause: Imbalanced data (91.3% legit vs 8.7% fraud) biases the model toward "legit".
# Next step: Apply balancing techniques (Oversampling, Undersampling, SMOTE) to improve recall.

In [15]:
# RandomUnderSampler --> removes random instances from the majority class (No Fraud)
# until it matches the size of the minority class (Fraud)
rus = RandomUnderSampler(random_state=SEED)
X_train_under, y_train_under = rus.fit_resample(X_train_scaled_df, y_train)

print('Before:', y_train.value_counts())
print('After: ', y_train_under.value_counts())

undersampling_model = SGDClassifier(loss='log_loss', random_state=SEED)
undersampling_model.fit(X_train_under, y_train_under)  # Fit on the undersampled Train set

print("Model trained!")

Before: fraud
0.0    730214
1.0     69786
Name: count, dtype: int64
After:  fraud
0.0    69786
1.0    69786
Name: count, dtype: int64
Model trained!


In [16]:
# Evaluate the model on both Train and Test, to check for overfitting
y_pred_train = undersampling_model.predict(X_train_under)   # train (undersampled)
y_pred_test = undersampling_model.predict(X_test_scaled_df) # test (untouched!)

accuracy_train = accuracy_score(y_train_under, y_pred_train)
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()

# Classification Report (Test)
# ----------------------------
print(classification_report(y_test, y_pred_test, target_names=['No Fraud', 'Fraud'], zero_division=0))

Accuracy (Train): 93.95%

              precision    recall  f1-score   support

    No Fraud       0.99      0.93      0.96    182383
       Fraud       0.56      0.95      0.70     17617

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.83    200000
weighted avg       0.96      0.93      0.94    200000



In [17]:
# Analysis 2
# Model 2 - Logistic Regression with Undersampling
# Accuracy: 93.95% - slightly lower than the imbalanced model but more meaningful
# No Fraud (class 0): Precision = 0.99, Recall = 0.93 - still very good at identifying legit transactions
# Fraud (class 1): Recall = 0.95 - massive improvement from 0.60, now catches 95% of fraud cases
# Fraud (class 1): Precision = 0.56 - more false alarms, only right 56% of the time when flagging fraud
# Key improvement: Recall for fraud jumped from 0.60 to 0.95 - 4 out of 10 missed frauds is now only 1 out of 20
# Trade-off: Higher recall came at the cost of lower precision - more false alarms
# Undersampling works by removing legit transactions to balance the dataset
# Downside: We lost a lot of training data (from 200,000 to ~35,000 rows) which could hurt generalization
# Next step: Try Oversampling to balance without losing data

In [18]:
# RandomOverSampler --> duplicates random instances from the minority class (Fraud)
# until it matches the size of the majority class (No Fraud)
ros = RandomOverSampler(random_state=SEED)
X_train_over, y_train_over = ros.fit_resample(X_train_scaled_df, y_train)

print('Before:', y_train.value_counts())
print('After: ', y_train_over.value_counts())

oversampling_model = SGDClassifier(loss='log_loss', random_state=SEED)
oversampling_model.fit(X_train_over, y_train_over)  # Fit on the oversampled Train set

print("Model trained!")

Before: fraud
0.0    730214
1.0     69786
Name: count, dtype: int64
After:  fraud
0.0    730214
1.0    730214
Name: count, dtype: int64
Model trained!


In [19]:
# Evaluate the model on both Train and Test, to check for overfitting
y_pred_train = oversampling_model.predict(X_train_over)     # train (oversampled)
y_pred_test = oversampling_model.predict(X_test_scaled_df)  # test (untouched!)

accuracy_train = accuracy_score(y_train_over, y_pred_train)
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()

# Classification Report (Test)
# ----------------------------
print(classification_report(y_test, y_pred_test, target_names=['No Fraud', 'Fraud'], zero_division=0))

Accuracy (Train): 94.23%

              precision    recall  f1-score   support

    No Fraud       0.99      0.93      0.96    182383
       Fraud       0.58      0.95      0.72     17617

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



In [ ]:
# Analysis 3
# Model 3 - Logistic Regression with Oversampling
# Accuracy: 93% - similar to undersampling model
# No Fraud (class 0): Precision = 0.99, Recall = 0.93 - identical to undersampling model
# Fraud (class 1): Recall = 0.95 - same as undersampling, catches 95% of fraud cases
# Fraud (class 1): Precision = 0.58 - slightly better than undersampling (0.56)
# F1 for Fraud = 0.72 - slight improvement over undersampling (0.70)
# Key advantage over Undersampling: No data was lost - all 200,000 training rows were kept
# Oversampling works by duplicating fraud cases to balance the dataset
# Downside: Duplicating data can cause overfitting - model may memorize duplicated fraud cases
# Overall: Both Oversampling and Undersampling significantly improved recall vs imbalanced model
# Next step: Try SMOTE which creates synthetic fraud cases instead of duplicating existing ones

In [20]:
# SMOTE --> generates synthetic samples for the minority class (Fraud) via KNN interpolation,
# instead of duplicating existing points, until it matches the size of the majority class
sm = SMOTE(random_state=SEED)
X_train_smote, y_train_smote = sm.fit_resample(X_train_scaled_df, y_train)

print('Before:', y_train.value_counts())
print('After: ', y_train_smote.value_counts())

smote_model = SGDClassifier(loss='log_loss', random_state=SEED)
smote_model.fit(X_train_smote, y_train_smote)  # Fit on the SMOTE-resampled Train set

print("Model trained!")

Before: fraud
0.0    730214
1.0     69786
Name: count, dtype: int64
After:  fraud
0.0    730214
1.0    730214
Name: count, dtype: int64
Model trained!


In [21]:
y_pred_train = smote_model.predict(X_train_smote)
y_pred_test = smote_model.predict(X_test_scaled_df)

accuracy_train = accuracy_score(y_train_smote, y_pred_train)
print(f'Accuracy (Train): {accuracy_train * 100:.2f}%')
print()

print(classification_report(y_test, y_pred_test, target_names=['No Fraud', 'Fraud'], zero_division=0))

Accuracy (Train): 94.21%

              precision    recall  f1-score   support

    No Fraud       0.99      0.93      0.96    182383
       Fraud       0.58      0.95      0.72     17617

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



In [ ]:
# Analysis 4
# Model 4 - Logistic Regression with SMOTE
# Accuracy: 93% - consistent with other balanced models
# No Fraud (class 0): Precision = 0.99, Recall = 0.93 - identical across all balanced models
# Fraud (class 1): Recall = 0.95 - same as Oversampling and Undersampling
# Fraud (class 1): Precision = 0.58 - same as Oversampling
# F1 for Fraud = 0.72 - same as Oversampling
# SMOTE advantage: Creates synthetic fraud cases instead of duplicating - less risk of overfitting
# For this dataset, SMOTE and Oversampling performed identically

In [ ]:
# ============================================================
# OVERALL CONCLUSION
# ============================================================
# Model 1 - Imbalanced:    Recall=0.60, Precision=0.90 - high precision but misses 40% of fraud
# Model 2 - Undersampling: Recall=0.95, Precision=0.56 - best recall but loses training data
# Model 3 - Oversampling:  Recall=0.95, Precision=0.58 - same recall, keeps all data
# Model 4 - SMOTE:         Recall=0.95, Precision=0.58 - same as oversampling, less overfitting risk
# 
# Best model for fraud detection: SMOTE or Oversampling
# Reason: Recall is the most important metric for fraud detection.
# Missing real fraud (low recall) is far more costly than false alarms (low precision).
# All balancing techniques significantly improved recall from 0.60 to 0.95.
# SMOTE is theoretically the most robust as it generates diverse synthetic data.